# SQL Surgeon Tester Notebook (No Clone Required)

Use this notebook to evaluate the hosted SQL Surgeon environment with your own model.

- Hosted env API: `https://aryadeep232006-sqlsurgeon.hf.space`
- Required API flow: `POST /reset` -> `POST /step` -> `GET /state`
- In the config cell, set `PROVIDER` to `hf`, `openai`, or `custom`.
- Credentials are pulled from env vars first (`HF_TOKEN`, `OPENAI_API_KEY`, or `MODEL_API_KEY`), then prompted interactively if missing.

In [ ]:
!pip -q install requests openai

In [ ]:
import os
import json
import requests
from openai import OpenAI

# --- Environment API (hosted SQL Surgeon) ---
BASE_URL = "https://aryadeep232006-sqlsurgeon.hf.space"
TASK_ID = "filter_scan"
MAX_STEPS = 5

# --- Provider/model config ---
# Choose one: "hf", "openai", "custom"
PROVIDER = "hf"

if PROVIDER == "hf":
    MODEL_API_BASE = "https://router.huggingface.co/v1"
    MODEL_NAME = "Qwen/Qwen2.5-72B-Instruct"
    MODEL_API_KEY = os.getenv("HF_TOKEN", "")
elif PROVIDER == "openai":
    MODEL_API_BASE = "https://api.openai.com/v1"
    MODEL_NAME = "gpt-4.1-mini"
    MODEL_API_KEY = os.getenv("OPENAI_API_KEY", "")
elif PROVIDER == "custom":
    # Any OpenAI-compatible endpoint
    MODEL_API_BASE = os.getenv("MODEL_API_BASE", "https://your-openai-compatible-endpoint/v1")
    MODEL_NAME = os.getenv("MODEL_NAME", "your-model-name")
    MODEL_API_KEY = os.getenv("MODEL_API_KEY", "")
else:
    raise ValueError("PROVIDER must be one of: hf, openai, custom")

if not MODEL_API_KEY:
    prompt = f"Paste API key for provider '{PROVIDER}' (or set env var first): "
    MODEL_API_KEY = input(prompt).strip()

if not MODEL_API_KEY:
    raise ValueError(
        "Missing API key. Set HF_TOKEN (hf), OPENAI_API_KEY (openai), or MODEL_API_KEY (custom)."
    )

client = OpenAI(base_url=MODEL_API_BASE, api_key=MODEL_API_KEY)
print("Provider:", PROVIDER)
print("Configured model:", MODEL_NAME)
print("Model API base:", MODEL_API_BASE)
print("Env base URL:", BASE_URL)

In [ ]:
def env_reset(task_id: str):
    resp = requests.post(f"{BASE_URL}/reset", json={"task_id": task_id}, timeout=60)
    resp.raise_for_status()
    return resp.json()


def env_step(action_payload: dict):
    # Required shape: {"action": {...}}
    resp = requests.post(f"{BASE_URL}/step", json={"action": action_payload}, timeout=60)
    resp.raise_for_status()
    return resp.json()


def build_prompt(observation: dict):
    return f"""
You are optimizing SQL for correctness first, then speed.

Task: {observation.get('task_id', '')}
Description: {observation.get('task_description', '')}
Original query:\n{observation.get('original_query', '')}
Schema DDL:\n{observation.get('schema_ddl', '')}

Return ONLY valid JSON with keys:
- action_type: one of [\"think\", \"explain\", \"schema\", \"submit\"]
- query: SQL string (required for submit/explain)
- thoughts: brief rationale
- confidence: number between 0 and 1
""".strip()


def decide_action(observation: dict):
    """Swap this logic with any agent/model framework you prefer."""
    prompt = build_prompt(observation)
    completion = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.2,
        messages=[
            {"role": "system", "content": "Return strict JSON only."},
            {"role": "user", "content": prompt},
        ],
    )
    text = completion.choices[0].message.content.strip()

    # Try direct JSON first
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        # Very small fallback: submit original query if model response is malformed.
        parsed = {
            "action_type": "submit",
            "query": observation.get("original_query", ""),
            "thoughts": "Fallback submit due to non-JSON model output.",
            "confidence": 0.2,
        }

    return {
        "action_type": str(parsed.get("action_type", "think")).strip().lower(),
        "query": parsed.get("query", "") or "",
        "thoughts": parsed.get("thoughts", ""),
        "confidence": float(parsed.get("confidence", 0.5)),
    }

In [ ]:
result = env_reset(TASK_ID)
obs = result["observation"]

print("Episode started")
print("Task:", obs.get("task_id"))
print("Description:", obs.get("task_description", "")[:200], "...")

for i in range(MAX_STEPS):
    action = decide_action(obs)
    step_result = env_step(action)
    obs = step_result["observation"]
    print(f"step={i+1} action={action['action_type']} reward={step_result['reward']:.3f} done={step_result['done']}")
    if step_result["done"]:
        break

print("Final reward:", step_result["reward"])
print("Final observation keys:", list(obs.keys())[:15])